# 中国地方政府债券利差高级计量经济学框架
## Advanced Econometric Framework for China Local Government Bond Spread Analysis

---

**作者**: Quinn Liu  
**GitHub**: https://github.com/quinnmacro  
**LinkedIn**: https://www.linkedin.com/in/liulu-math  

**核心理念**: 模型锦标赛 (Model Tournament) - 不依赖单一模型，让数据选择最优动态特征

---

### 分析框架三大支柱:

1. **波动率建模锦标赛** (GARCH/EGARCH/GJR-GARCH) - 捕捉不对称效应
2. **卡尔曼滤波器** - 从噪音中提取真实信号
3. **极值理论** (EVT) - 尾部风险量化

In [ ]:
# 环境配置
import sys
sys.path.insert(0, '..')  # 添加父目录到路径

from src import (
    DataEngine,
    VolatilityModeler,
    KalmanSignalExtractor,
    EVTRiskAnalyzer,
    plot_signal_trend,
    plot_volatility_structure,
    plot_tail_risk,
    print_var_comparison,
    generate_strategic_report
)

import warnings
warnings.filterwarnings('ignore')

print("✓ 所有模块加载完成")

---
## 第一部分: 配置与数据加载

In [ ]:
# 全局配置
CONFIG = {
    'SOURCE': 'MOCK',  # 'WIND_EDB' 或 'MOCK'
    'TICKER': 'M0017142',
    'START_DATE': '2018-01-01',
    'END_DATE': '2025-12-31',
    'MAD_THRESHOLD': 5.0,
    'GARCH_P': 1,
    'GARCH_Q': 1,
    'VaR_CONFIDENCE': 0.99,
    'EVT_THRESHOLD_PERCENTILE': 0.95
}

print(f"数据源: {CONFIG['SOURCE']}")
print(f"分析周期: {CONFIG['START_DATE']} 至 {CONFIG['END_DATE']}")

In [ ]:
# 数据加载与清洗
engine = DataEngine(CONFIG)
raw_data = engine.load_data()
clean_data = engine.clean_data()
returns = engine.get_returns()

print(f"\n数据统计摘要:")
print(clean_data['spread'].describe())

---
## 第二部分: 模型锦标赛

In [ ]:
# 模块 A: GARCH 锦标赛
vol_modeler = VolatilityModeler(returns, p=CONFIG['GARCH_P'], q=CONFIG['GARCH_Q'])
winner_model = vol_modeler.run_tournament()
winner_volatility = vol_modeler.get_conditional_volatility(winner_model)

In [ ]:
# 模块 B: 卡尔曼滤波
kalman = KalmanSignalExtractor(clean_data['spread'])
smoothed_spread = kalman.fit()
signal_deviation = kalman.get_signal_deviation()

print(f"\n当前偏离度: {signal_deviation.iloc[-1]:.2f} σ")

In [ ]:
# 模块 C: 极值理论
evt_analyzer = EVTRiskAnalyzer(
    returns,
    threshold_percentile=CONFIG['EVT_THRESHOLD_PERCENTILE'],
    confidence=CONFIG['VaR_CONFIDENCE']
)
evt_analyzer.fit_gpd()
evt_var = evt_analyzer.calculate_var()

---
## 第三部分: 可视化

In [ ]:
# 图表 1: 信号与趋势
fig1 = plot_signal_trend(clean_data, smoothed_spread, signal_deviation)
fig1.show()

In [ ]:
# 图表 2: 波动率结构
fig2 = plot_volatility_structure(winner_volatility, winner_model)
fig2.show()

In [ ]:
# 图表 3: 尾部风险锥
fig3, evt_var_out, empirical_var = plot_tail_risk(returns, evt_var, CONFIG['VaR_CONFIDENCE'])
fig3.show()
print_var_comparison(evt_var_out, empirical_var)

---
## 第四部分: 战略报告

In [ ]:
# 生成战略分析报告
generate_strategic_report(
    winner_model=winner_model,
    vol_modeler=vol_modeler,
    clean_data=clean_data,
    smoothed_spread=smoothed_spread,
    signal_deviation=signal_deviation,
    winner_volatility=winner_volatility,
    evt_var=evt_var
)